In [29]:
import os
import logging
import sys
from typing import List, Optional

In [30]:
from datasets import load_dataset

# Vector stores and embeddings
from langchain_community.vectorstores import Chroma

# HTTP + model dependencies
import httpx
from transformers import AutoModel, AutoTokenizer
import torch

# Core building blocks
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [31]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("rag_pipeline")


In [32]:
# ===========================================================================
# SECTION 1 -- CONFIGURATION
# All tunable parameters are centralized here for clean separation of concerns
# ===========================================================================


In [ ]:

class RAGConfig:
    """
    Centralized configuration for the RAG pipeline.

    Keeping all hyper-parameters in one place makes it trivial to swap
    components (different LLM, different chunk size, different collection)
    without hunting through business logic.
    """

    # Dataset settings
    DATASET_NAME: str = "rajpurkar/squad_v2"
    DATASET_SPLIT: str = "train"
    MAX_DOCS: int = 500           # Ingest first 500 unique contexts to keep demo fast

    # Text splitting -- RecursiveCharacterTextSplitter walks a priority list of
    # separators (\n\n, \n, space, "") ensuring semantically coherent chunks
    CHUNK_SIZE: int = 512         # Characters per chunk (tune based on LLM context window)
    CHUNK_OVERLAP: int = 64       # Overlap preserves cross-boundary context

    # Embeddings -- all-MiniLM-L6-v2 is a 22M-param model, fast and high quality
    # for English semantic similarity. Runs fully locally.
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = "cpu"  # Change to "cuda" if GPU available

    # ChromaDB -- persist to disk so vectors survive process restarts
    CHROMA_PERSIST_DIR: str = "./chroma_db"
    CHROMA_COLLECTION: str = "squad_rag"

    # Azure AI Foundry / Azure OpenAI deployment settings
    AI_FOUNDRY_PROJECT_ENDPOINT: str = "https:"
    AI_FOUNDRY_DEPLOYMENT_NAME: str = "gpt-4.1"
    AI_FOUNDRY_API_VERSION: str = "2024-12-01-preview"
    AI_FOUNDRY_API_KEY: str = ""
    LLM_TEMPERATURE: float = 0.0
    LLM_MAX_TOKENS: int = 512

    # Retrieval -- top-k documents fetched per query; MMR adds diversity
    RETRIEVER_K: int = 4
    RETRIEVER_SEARCH_TYPE: str = "mmr"  # "similarity" | "mmr" | "similarity_score_threshold"
    MMR_FETCH_K: int = 20               # Candidate pool for MMR re-ranking

    # Prompt template -- explicit instruction keeps the LLM grounded in retrieved context
    RAG_PROMPT_TEMPLATE: str = """You are an expert question-answering assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present in the context, respond with:
"I could not find a relevant answer in the provided context."

Context:
{context}

Question: {question}

Provide a concise, accurate answer:"""


In [34]:

# ===========================================================================
# SECTION 2 -- DATA INGESTION
# Load SQuAD v2 from HuggingFace Hub, deduplicate contexts, wrap as Documents
# ===========================================================================

def load_squad_documents(config: RAGConfig) -> List[Document]:
    """
    Load SQuAD v2 from HuggingFace datasets and convert to LangChain Documents.

    SQuAD v2 structure per row:
        - id        : unique example id
        - title     : Wikipedia article title
        - context   : passage paragraph (this becomes our retrieval unit)
        - question  : question about the passage
        - answers   : dict with 'text' list and 'answer_start' list

    We deduplicate on `context` because many questions share the same passage.
    Each Document carries metadata (title, source) for provenance tracking.

    Args:
        config: RAGConfig instance with dataset parameters.

    Returns:
        List of LangChain Document objects ready for splitting.

    Raises:
        RuntimeError: If the dataset cannot be loaded from HuggingFace Hub.
    """
    logger.info("Loading dataset '%s' split='%s' ...", config.DATASET_NAME, config.DATASET_SPLIT)

    try:
        # trust_remote_code=False is the secure default; SQuAD does not need it
        dataset = load_dataset(config.DATASET_NAME, split=config.DATASET_SPLIT)
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load dataset '{config.DATASET_NAME}'. "
            "Ensure you have internet access and the `datasets` package installed."
        ) from exc

    logger.info("Raw dataset size: %d rows", len(dataset))

    # Deduplicate by context text so we don't embed the same passage repeatedly.
    # Using a dict preserves insertion order (Python 3.7+) and keeps the first
    # occurrence of each unique context along with its metadata.
    seen_contexts: dict = {}
    for row in dataset:
        ctx = row["context"].strip()
        if ctx not in seen_contexts:
            seen_contexts[ctx] = {
                "title": row["title"],
                "source": config.DATASET_NAME,
            }
        if len(seen_contexts) >= config.MAX_DOCS:
            break

    # Convert to LangChain Documents
    documents: List[Document] = [
        Document(page_content=ctx, metadata=meta)
        for ctx, meta in seen_contexts.items()
    ]

    logger.info("Unique context documents after deduplication: %d", len(documents))
    return documents


In [35]:

# ===========================================================================
# SECTION 3 -- TEXT SPLITTING
# Chunk documents into segments the embedding model and LLM can handle well
# ===========================================================================

def split_documents(documents: List[Document], config: RAGConfig) -> List[Document]:
    """
    Split raw documents into semantically coherent chunks using
    RecursiveCharacterTextSplitter.

    This splitter attempts to split on paragraph breaks first (\n\n), then
    newlines, then spaces, and finally individual characters as a last resort.
    The overlap ensures that a sentence straddling two chunks is partially
    present in both, preventing context loss at boundaries.

    Args:
        documents : List of raw Documents from the ingestion step.
        config    : RAGConfig with CHUNK_SIZE and CHUNK_OVERLAP values.

    Returns:
        List of split Document chunks with inherited metadata.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        length_function=len,        # character-based; use tiktoken for token-based
        add_start_index=True,       # adds 'start_index' to metadata for traceability
    )

    chunks = splitter.split_documents(documents)
    logger.info(
        "Split %d documents into %d chunks (size=%d, overlap=%d)",
        len(documents), len(chunks), config.CHUNK_SIZE, config.CHUNK_OVERLAP,
    )
    return chunks



In [36]:


# ===========================================================================
# SECTION 4 -- VECTOR STORE CONSTRUCTION
# Embed chunks and persist them in ChromaDB
# ===========================================================================

class LocalTransformerEmbeddings(Embeddings):
    """Minimal local embedding wrapper built directly on top of transformers."""

    def __init__(self, model_name: str, device: str = "cpu") -> None:
        resolved_device = device
        if device == "cuda" and not torch.cuda.is_available():
            logger.warning("CUDA requested for embeddings, but no GPU was detected. Falling back to CPU.")
            resolved_device = "cpu"

        self.model_name = model_name
        self.device = torch.device(resolved_device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def _embed(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []

        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {key: value.to(self.device) for key, value in encoded.items()}

        with torch.inference_mode():
            outputs = self.model(**encoded)

        token_embeddings = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
        pooled = (token_embeddings * attention_mask).sum(dim=1)
        pooled = pooled / attention_mask.sum(dim=1).clamp(min=1e-9)
        normalized = torch.nn.functional.normalize(pooled, p=2, dim=1)
        return normalized.cpu().tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed(texts)

    def embed_query(self, text: str) -> List[float]:
        return self._embed([text])[0]

def build_or_load_vectorstore(
    chunks: Optional[List[Document]],
    config: RAGConfig,
    force_rebuild: bool = False,
) -> Chroma:
    """
    Build a ChromaDB vector store from document chunks, or load an existing one.

    ChromaDB persists embeddings to disk at CHROMA_PERSIST_DIR. On subsequent
    runs when the store already exists, we skip re-embedding (expensive) unless
    force_rebuild=True is explicitly requested.

    The embedding model (all-MiniLM-L6-v2) produces 384-dimensional vectors.
    ChromaDB uses cosine similarity by default for nearest-neighbor search,
    which is appropriate for normalized sentence embeddings.

    Args:
        chunks        : Chunked Document list. Required if store does not exist.
        config        : RAGConfig with embedding and persistence settings.
        force_rebuild : If True, delete existing collection and rebuild from scratch.

    Returns:
        A Chroma vector store instance ready for retrieval.

    Raises:
        ValueError : If chunks is None and no persistent store exists.
    """
    logger.info("Initializing local transformer embedding model: %s", config.EMBEDDING_MODEL)
    embeddings = LocalTransformerEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        device=config.EMBEDDING_DEVICE,
    )

    # Check if a persisted collection already exists
    collection_exists = (
        os.path.isdir(config.CHROMA_PERSIST_DIR)
        and any(os.scandir(config.CHROMA_PERSIST_DIR))
    )

    if collection_exists and not force_rebuild:
        logger.info(
            "Loading existing ChromaDB collection from '%s' ...",
            config.CHROMA_PERSIST_DIR,
        )
        vectorstore = Chroma(
            collection_name=config.CHROMA_COLLECTION,
            embedding_function=embeddings,
            persist_directory=config.CHROMA_PERSIST_DIR,
        )
        logger.info(
            "Loaded collection '%s' with %d vectors.",
            config.CHROMA_COLLECTION,
            vectorstore._collection.count(),     # internal count call
        )
        return vectorstore

    # Build a fresh vector store
    if chunks is None:
        raise ValueError(
            "chunks must be provided when building a new vector store. "
            "Either pass document chunks or set force_rebuild=False to load existing store."
        )

    logger.info(
        "Building ChromaDB collection '%s' from %d chunks ...",
        config.CHROMA_COLLECTION, len(chunks),
    )

    # Chroma.from_documents embeds all chunks and writes them to disk in batches
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=config.CHROMA_COLLECTION,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )

    logger.info(
        "Vector store built. Total vectors stored: %d",
        vectorstore._collection.count(),
    )
    return vectorstore


In [37]:

# ===========================================================================
# SECTION 5 -- RAG CHAIN CONSTRUCTION (LangChain Expression Language)
# ===========================================================================

def build_rag_chain(vectorstore: Chroma, config: RAGConfig):
    """
    Assemble the end-to-end RAG chain using LangChain Expression Language (LCEL).

    LCEL uses the pipe operator (|) to compose Runnables into a declarative
    execution graph. Each component is lazy and supports async, streaming,
    and batching out of the box.

    Chain architecture:
        {question} --> retriever (ChromaDB MMR) --> format_docs --> prompt
                   --> Azure AI Foundry chat completions API --> str answer

    The retriever uses MMR (Maximal Marginal Relevance) to balance relevance
    and diversity: it fetches MMR_FETCH_K candidates from the index, then
    selects RETRIEVER_K final documents that are both relevant to the query
    and mutually dissimilar, reducing redundant context in the prompt.

    Args:
        vectorstore : Populated Chroma vector store.
        config      : RAGConfig with LLM and retriever settings.

    Returns:
        A compiled LCEL Runnable chain. Call with {"question": "..."}.

    Raises:
        EnvironmentError : If the Azure AI Foundry settings are missing.
    """
    required_settings = {
        "AI_FOUNDRY_PROJECT_ENDPOINT": config.AI_FOUNDRY_PROJECT_ENDPOINT,
        "AI_FOUNDRY_DEPLOYMENT_NAME": config.AI_FOUNDRY_DEPLOYMENT_NAME,
        "AI_FOUNDRY_API_VERSION": config.AI_FOUNDRY_API_VERSION,
        "AI_FOUNDRY_API_KEY": config.AI_FOUNDRY_API_KEY,
    }
    missing_settings = [name for name, value in required_settings.items() if not value]
    if missing_settings:
        raise EnvironmentError(
            "Missing Azure AI Foundry configuration values: " + ", ".join(missing_settings)
        )

    # Build the retriever from the vector store with MMR search
    retriever = vectorstore.as_retriever(
        search_type=config.RETRIEVER_SEARCH_TYPE,
        search_kwargs={
            "k": config.RETRIEVER_K,
            "fetch_k": config.MMR_FETCH_K,
        },
    )

    # Construct the prompt template from the config string
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT_TEMPLATE)

    # Helper: concatenate retrieved Document page_content with source attribution
    def format_docs(docs: List[Document]) -> str:
        """
        Flatten retrieved documents into a single context string.

        Each document is separated by a horizontal rule and annotated with
        its source title so the model can reference provenance in its answer.
        """
        formatted_parts = []
        for i, doc in enumerate(docs, start=1):
            title = doc.metadata.get("title", "Unknown")
            formatted_parts.append(
                f"[Source {i} -- {title}]\n{doc.page_content.strip()}"
            )
        return "\n\n---\n\n".join(formatted_parts)

    def call_ai_foundry(prompt_value) -> str:
        role_map = {
            "system": "system",
            "human": "user",
            "ai": "assistant",
        }
        messages = []
        for message in prompt_value.to_messages():
            role = role_map.get(message.type, message.type)
            messages.append({"role": role, "content": message.content})

        endpoint = config.AI_FOUNDRY_PROJECT_ENDPOINT.rstrip("/")
        url = (
            f"{endpoint}/openai/deployments/{config.AI_FOUNDRY_DEPLOYMENT_NAME}"
            f"/chat/completions?api-version={config.AI_FOUNDRY_API_VERSION}"
        )
        response = httpx.post(
            url,
            headers={
                "Content-Type": "application/json",
                "api-key": config.AI_FOUNDRY_API_KEY,
            },
            json={
                "messages": messages,
                "temperature": config.LLM_TEMPERATURE,
                "max_tokens": config.LLM_MAX_TOKENS,
            },
            timeout=60.0,
        )
        if response.is_error:
            raise RuntimeError(
                f"Azure AI Foundry request failed with {response.status_code}: {response.text}"
            )

        payload = response.json()
        return payload["choices"][0]["message"]["content"].strip()

    # LCEL chain: retriever runs in parallel with the passthrough for the question,
    # then context and question are fed into the prompt and Foundry endpoint.
    rag_chain = (
        {
            "context": retriever | format_docs,   # retrieve docs then format them
            "question": RunnablePassthrough(),     # pass question through unchanged
        }
        | prompt
        | RunnableLambda(call_ai_foundry)
    )

    logger.info("RAG chain assembled successfully.")
    return rag_chain, retriever


In [38]:


# ===========================================================================
# SECTION 6 -- QUERY INTERFACE
# Run a query and optionally display retrieved source documents
# ===========================================================================

def query_rag(
    chain,
    retriever,
    question: str,
    show_sources: bool = True,
) -> str:
    """
    Run a question through the RAG pipeline and return the answer.

    Also optionally prints the source documents used for grounding so the
    user can verify factual provenance -- critical for production trust.

    Args:
        chain        : Compiled LCEL RAG chain.
        retriever    : The same retriever used in the chain (for source display).
        question     : Natural language question string.
        show_sources : If True, print retrieved source chunks.

    Returns:
        Answer string generated by the LLM.
    """
    logger.info("Processing query: '%s'", question)

    # Retrieve source documents for display (independent call, same retriever)
    if show_sources:
        source_docs = retriever.invoke(question)
        print("\n" + "=" * 70)
        print(f"QUERY: {question}")
        print("=" * 70)
        print(f"\nRETRIEVED SOURCES ({len(source_docs)} documents):")
        for i, doc in enumerate(source_docs, start=1):
            title = doc.metadata.get("title", "Unknown")
            snippet = doc.page_content[:200].replace("\n", " ").strip()
            print(f"\n  [{i}] Title: {title}")
            print(f"      Excerpt: {snippet}...")

    # Run the full chain to get the LLM answer
    answer = chain.invoke(question)

    if show_sources:
        print("\nANSWER:")
        print(answer)
        print("=" * 70 + "\n")

    return answer


In [39]:


# ===========================================================================
# SECTION 7 -- MAIN ENTRYPOINT
# Orchestrates: ingest -> split -> embed -> store -> query demo
# ===========================================================================

def main() -> None:
    """
    End-to-end RAG pipeline orchestrator.

    On first run  : downloads SQuAD v2, embeds chunks, persists to ChromaDB.
    On second run : loads the persisted ChromaDB collection (fast, no re-embedding).
    Both runs     : executes a set of representative demo queries.
    """
    config = RAGConfig()

    logger.info("Starting RAG pipeline ...")

    # Determine if we need to build or just load the vector store
    store_exists = (
        os.path.isdir(config.CHROMA_PERSIST_DIR)
        and any(os.scandir(config.CHROMA_PERSIST_DIR))
    )

    if not store_exists:
        # First run: full ingestion pipeline
        logger.info("No existing vector store found. Running full ingestion pipeline.")

        raw_docs = load_squad_documents(config)
        chunks = split_documents(raw_docs, config)
        vectorstore = build_or_load_vectorstore(chunks, config, force_rebuild=False)
    else:
        # Subsequent runs: skip embedding, load from disk
        logger.info("Existing vector store detected. Skipping ingestion.")
        vectorstore = build_or_load_vectorstore(None, config, force_rebuild=False)

    # Build the RAG chain
    rag_chain, retriever = build_rag_chain(vectorstore, config)

    # Demo queries -- these are representative of SQuAD v2 question styles,
    # covering factual recall, named entities, and descriptive questions
    demo_questions = [
        "Where did the Normans come from originally?",
        "What was the main cause of the Byzantine Empire's decline?",
        "Who invented the telephone?",
        "What is the significance of the Magna Carta?",
        "What architectural style is Notre-Dame Cathedral built in?",
    ]

    logger.info("Running %d demo queries ...", len(demo_questions))
    for question in demo_questions:
        query_rag(rag_chain, retriever, question, show_sources=True)

    logger.info("RAG pipeline demo completed successfully.")



In [40]:

if __name__ == "__main__":
    main()


2026-03-22 22:13:26 | INFO     | rag_pipeline | Starting RAG pipeline ...
2026-03-22 22:13:26 | INFO     | rag_pipeline | Existing vector store detected. Skipping ingestion.
2026-03-22 22:13:26 | INFO     | rag_pipeline | Initializing local transformer embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-03-22 22:13:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-22 22:13:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-22 22:13:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-22 22:13:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 614.78it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-03-22 22:13:28 | INFO     | rag_pipeline | Loading existing ChromaDB collection from './chroma_db' ...
2026-03-22 22:13:28 | INFO     | rag_pipeline | Loaded collection 'squad_rag' with 920 vectors.
2026-03-22 22:13:28 | INFO     | rag_pipeline | RAG chain assembled successfully.
2026-03-22 22:13:28 | INFO     | rag_pipeline | Running 5 demo queries ...
2026-03-22 22:13:28 | INFO     | rag_pipeline | Processing query: 'Where did the Normans come from originally?'

QUERY: Where did the Normans come from originally?

RETRIEVED SOURCES (4 documents):

  [1] Title: New_York_City
      Excerpt: single-family homes are common in various architectural styles such as Tudor Revival and Victorian....

  [2] Title: New_York_City
      Excerpt: that it might represent an oceanic tributary. When the river narrowed and was no longer saline, he realized it was not a maritime passage and sailed back downriver. He made a ten-day exploration of th...

  [3] Title: 2008_Sichuan_earthquake
      Exce